# Probability Distributions

Companion notebook for the [Probability Distributions](https://ml-viz-ruby.vercel.app/courses/probability-statistics/02-probability-distributions) lesson.

We implement the exact math from the lesson: the PMF/PDF distinction, deriving and **empirically verifying** $\mathbb{E}[X]$ and $\mathrm{Var}(X)$ for the Bernoulli and Gaussian, checking the Gaussian normalization integral, and fitting a Bernoulli by maximum likelihood.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

BRAND, TEAL, ORANGE = '#6366f1', '#2dd4bf', '#f97316'

_trapz = getattr(np, "trapezoid", None) or np.trapz  # numpy renamed trapz -> trapezoid in 2.0


## Intuition — the vocabulary of uncertainty

A handful of named distributions describe almost everything ML models about
uncertainty. **Bernoulli/Binomial** count successes (a classifier's yes/no). The
**Gaussian** is the default for continuous noise and the shape the **Central Limit
Theorem** pushes averages toward. **Poisson** counts rare events. The one distinction to
nail first: a discrete **PMF** gives an actual probability at each value, while a
continuous **PDF** gives *density* — probability per unit length — whose value can exceed
1 even though the area under it is exactly 1. We implement each distribution's formula by
hand, verify its mean/variance by heavy sampling, and cross-check against `scipy.stats`.

## 1. PMF vs PDF

A **PMF** (discrete) gives an actual probability at each value: bars that sum to 1, each $\le 1$.
A **PDF** (continuous) gives *density* (probability per unit length): the area under the curve is 1, but the curve itself can exceed 1.

We illustrate both, including a narrow Gaussian whose peak is above 1 to drive home that a density is **not** a probability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# PMF: Bernoulli(0.7) -- two bars that sum to 1
ax = axes[0]
p = 0.7
ax.bar([0, 1], [1 - p, p], width=0.25, color=[ORANGE, BRAND], alpha=0.9)
ax.set_xticks([0, 1]); ax.set_xticklabels(['0 (failure)', '1 (success)'])
ax.set_ylim(0, 1)
ax.set_title('PMF: Bernoulli(0.7)  ->  bars sum to {:.1f}'.format((1 - p) + p))
ax.set_ylabel('probability  P(X=k)')

# PDF: a narrow Gaussian whose density peak is > 1
ax = axes[1]
x = np.linspace(-1.5, 1.5, 400)
sigma = 0.1
peak = stats.norm.pdf(0, 0, sigma)
ax.plot(x, stats.norm.pdf(x, 0, sigma), color=TEAL, lw=2)
ax.axhline(1.0, color='#94a3b8', ls='--', lw=1)
ax.set_title('PDF: N(0, 0.1^2)  ->  peak density = {:.2f}  (> 1!)'.format(peak))
ax.set_ylabel('density  p(x)')

for ax in axes.flat:
    ax.grid(True, alpha=0.2)
plt.suptitle('A density is not a probability', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

# The area under the PDF still integrates to 1:
area = _trapz(stats.norm.pdf(x, 0, sigma), x)
print('Area under the narrow-Gaussian PDF over [-1.5, 1.5]: {:.4f}'.format(area))

**What to notice:** the Bernoulli bars sum to 1 (a PMF *is* probability), but the narrow
Gaussian's peak density is well **above 1** — yet its area integrates to `1.00`. Density
is not probability; only the *area* over an interval is. Confusing the two is the single
most common distribution mistake.

## 2. Key distributions at a glance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Bernoulli for several p
ax = axes[0, 0]
for pp, color in [(0.3, BRAND), (0.5, TEAL), (0.7, ORANGE)]:
    ax.bar([0, 1], [1 - pp, pp], width=0.2, alpha=0.8, color=color,
           label='p={}'.format(pp), align='center')
ax.set_xticks([0, 1]); ax.set_xticklabels(['0 (failure)', '1 (success)'])
ax.set_title('Bernoulli PMF'); ax.legend()

# Gaussian for several (mu, sigma)
ax = axes[0, 1]
x = np.linspace(-6, 10, 400)
for mu, sig, color in [(0, 1, BRAND), (3, 1.5, ORANGE), (0, 2, TEAL)]:
    ax.plot(x, stats.norm.pdf(x, mu, sig), color=color, lw=2,
            label='N({}, {}^2)'.format(mu, sig))
ax.set_title('Gaussian PDF'); ax.legend()

# Central Limit Theorem
rng = np.random.default_rng(42)
ax = axes[1, 0]
for n, color in [(1, BRAND), (5, ORANGE), (30, TEAL)]:
    means = rng.uniform(0, 1, size=(5000, n)).mean(axis=1)
    ax.hist(means, bins=50, alpha=0.5, density=True, color=color, label='n={}'.format(n))
ax.set_title('Central Limit Theorem\n(mean of n Uniform[0,1] samples)')
ax.legend()

# 2D Gaussian contours
ax = axes[1, 1]
grid = np.linspace(-3, 3, 120)
XX, YY = np.meshgrid(grid, grid)
xy = np.column_stack([XX.ravel(), YY.ravel()])
for mu, cov, color, label in [
    ([0, 0], [[1, 0.8], [0.8, 1]], BRAND, 'rho=0.8'),
    ([0, 0], [[1, -0.5], [-0.5, 1]], ORANGE, 'rho=-0.5'),
]:
    density = stats.multivariate_normal(mu, cov).pdf(xy).reshape(120, 120)
    ax.contour(grid, grid, density, levels=5, colors=[color], alpha=0.7)
    ax.plot([], [], color=color, label=label)
ax.set_aspect('equal'); ax.set_title('2D Gaussian: different covariances')
ax.legend()

for ax in axes.flat:
    ax.grid(True, alpha=0.2)
plt.suptitle('Key Distributions in Machine Learning', y=1.01, fontsize=13)
plt.tight_layout(); plt.show()

**What to notice:** four workhorses in one view. Bernoulli shifts mass between 0 and 1
with `p`; the Gaussian slides with `μ` and widens with `σ`; the **CLT** panel is the
striking one — averaging just 30 *uniform* samples already yields a bell curve; and the
2-D Gaussian's covariance tilts its contours (positive vs negative correlation).

## 3. Verifying E[X] and Var(X) against the closed forms

The lesson derives, from first principles:

- **Bernoulli(p):** $\mathbb{E}[X] = p$, $\mathrm{Var}(X) = p(1-p)$.
- **Gaussian($\mu,\sigma^2$):** $\mathbb{E}[X] = \mu$, $\mathrm{Var}(X) = \sigma^2$.
- **Uniform[a,b]:** $\mathbb{E}[X] = \tfrac{a+b}{2}$, $\mathrm{Var}(X) = \tfrac{(b-a)^2}{12}$.

Now sample heavily and confirm the empirical moments match.

In [ ]:
rng = np.random.default_rng(0)
N = 200000

distributions = [
    ('Bernoulli(0.7)', rng.binomial(1, 0.7, N), 0.7, 0.7 * 0.3),
    ('Gaussian(2, 4)', rng.normal(2, 2, N),      2.0, 4.0),       # sigma=2 -> var=4
    ('Uniform[0, 1]',  rng.uniform(0, 1, N),     0.5, 1 / 12),
]

header = '{:18s}  {:>11s}  {:>11s}  {:>11s}  {:>11s}'.format(
    'Distribution', 'E[X] true', 'E[X] samp', 'Var true', 'Var samp')
print(header)
print('-' * len(header))
for name, s, true_mean, true_var in distributions:
    print('{:18s}  {:11.4f}  {:11.4f}  {:11.4f}  {:11.4f}'.format(
        name, true_mean, s.mean(), true_var, s.var()))

**What to notice:** every sampled mean and variance matches the closed form to ~3
decimals — `Bernoulli(0.7)` → `0.7, 0.21`, `Gaussian(2,4)` → `2, 4`, `Uniform[0,1]` →
`0.5, 0.083`. The formulas the lesson derived on paper are exactly what the data shows.

### Var(X) = E[X^2] - (E[X])^2, checked numerically

The lesson's shortcut for variance. We confirm both sides agree on a Gaussian sample.

In [ ]:
s = rng.normal(2, 2, N)
lhs = s.var()                       # E[(X-mu)^2]
rhs = (s ** 2).mean() - s.mean() ** 2  # E[X^2] - (E[X])^2
print('E[(X-mu)^2]            = {:.4f}'.format(lhs))
print('E[X^2] - (E[X])^2      = {:.4f}'.format(rhs))
print('true sigma^2           = {:.4f}'.format(4.0))

**What to notice:** the two variance expressions agree — `E[(X−μ)²]` and the shortcut
`E[X²] − (E[X])²` both land on ~4. The shortcut needs only two running sums, which is why
libraries use it.

## 4. The Gaussian normalization integral

The lesson shows $\int_{-\infty}^{\infty} e^{-x^2/2}\,dx = \sqrt{2\pi}$ via the polar-coordinate trick, so the full PDF (divided by $\sigma\sqrt{2\pi}$) integrates to 1. We verify both numerically.

In [ ]:
# Bare bell curve should integrate to sqrt(2*pi)
x = np.linspace(-12, 12, 100001)
bare = np.exp(-x ** 2 / 2)
print('integral of e^(-x^2/2)   = {:.6f}'.format(_trapz(bare, x)))
print('sqrt(2*pi)               = {:.6f}'.format(np.sqrt(2 * np.pi)))

# Full normalized PDF integrates to 1 for any (mu, sigma)
for mu, sigma in [(0, 1), (3, 1.5), (-2, 0.5)]:
    xx = np.linspace(mu - 12 * sigma, mu + 12 * sigma, 100001)
    pdf = np.exp(-(xx - mu) ** 2 / (2 * sigma ** 2)) / (sigma * np.sqrt(2 * np.pi))
    print('N({:>4}, {:>4}) total area = {:.6f}'.format(mu, sigma, _trapz(pdf, xx)))

**What to notice:** the bare `e^{−x²/2}` integrates to `√(2π) ≈ 2.5066`, and dividing by
`σ√(2π)` makes the full PDF integrate to exactly `1` for every `(μ, σ)`. That normalizer
is *why* the Gaussian has the constant it does.

## 5. Fitting a Bernoulli by maximum likelihood

The lesson derives the MLE for a coin: with $s$ heads in $n$ flips, the log-likelihood
$\ell(\theta) = s\log\theta + (n-s)\log(1-\theta)$ is maximized at $\hat\theta = s/n$.

We plot $\ell(\theta)$ for simulated data and confirm its peak lands on the head-fraction.

In [ ]:
rng = np.random.default_rng(7)
true_p = 0.7
n = 200
flips = rng.binomial(1, true_p, n)
s = flips.sum()

theta = np.linspace(0.001, 0.999, 999)
loglik = s * np.log(theta) + (n - s) * np.log(1 - theta)

mle_closed = s / n                 # derived in the lesson
mle_grid = theta[np.argmax(loglik)]  # numerical argmax for a sanity check

print('heads s = {} of n = {}'.format(s, n))
print('closed-form MLE  s/n   = {:.4f}'.format(mle_closed))
print('grid-search argmax     = {:.4f}'.format(mle_grid))
print('true p                 = {:.4f}'.format(true_p))

plt.figure(figsize=(8, 4.5))
plt.plot(theta, loglik, color=BRAND, lw=2)
plt.axvline(mle_closed, color=TEAL, ls='--', lw=1.5,
            label='MLE = s/n = {:.3f}'.format(mle_closed))
plt.axvline(true_p, color=ORANGE, ls=':', lw=1.5,
            label='true p = {:.2f}'.format(true_p))
plt.xlabel('theta'); plt.ylabel('log-likelihood  l(theta)')
plt.title('Bernoulli log-likelihood peaks at the head-fraction')
plt.grid(True, alpha=0.2); plt.legend()
plt.tight_layout(); plt.show()

**What to notice:** the log-likelihood is a smooth hill peaking at the head-fraction
`s/n`, and the closed-form MLE, the grid-search argmax, and the true `p` all coincide.
Maximum likelihood *is* "pick the parameter that makes the observed data most probable."

## 6. The library way — fitting distributions with `scipy.stats`

In practice you don't derive MLEs by hand: `scipy.stats.<dist>.fit(data)` returns the
maximum-likelihood parameters directly, and each frozen distribution exposes `.pdf`,
`.cdf`, `.rvs`, `.mean()`, `.var()`. The cell fits a Gaussian to samples, recovers `(μ,σ)`,
and asserts scipy's PDF equals our hand-written formula.

In [ ]:
from scipy import stats

data = rng.normal(2.0, 3.0, 5000)
mu_hat, sigma_hat = stats.norm.fit(data)          # MLE for a Gaussian, in one call
print(f'fit recovered mu = {mu_hat:.3f}, sigma = {sigma_hat:.3f}  (true 2.0, 3.0)')

xs = np.linspace(-10, 14, 9)
ours  = np.exp(-0.5 * ((xs - mu_hat)/sigma_hat)**2) / (sigma_hat * np.sqrt(2*np.pi))
scipy = stats.norm.pdf(xs, mu_hat, sigma_hat)
assert np.allclose(ours, scipy), "our Gaussian formula must equal scipy.stats.norm.pdf"
print('hand-written Gaussian PDF == scipy.stats.norm.pdf ✓')

**What to notice:** `fit` recovers `μ ≈ 2`, `σ ≈ 3` straight from the samples, and our
by-hand density matches `scipy.stats.norm.pdf` exactly. Understand the formula once, then
let `scipy.stats` (or `torch.distributions`) do the fitting and sampling in real code.

## 7. Gotchas & limitations

- **Density ≠ probability.** A PDF value can exceed 1; only its *integral* over an
  interval is a probability. `P(X = x) = 0` for any single point of a continuous variable.
- **Population vs sample variance.** NumPy's `.var()` divides by `N` (biased,
  population); the unbiased *sample* variance divides by `N−1` (`ddof=1`, Bessel's
  correction). The gap matters for small `n`.
- **Log-likelihood blows up at the boundary.** `log θ` → `−∞` as `θ → 0`; MLE code clips
  or works in log-space.
- **The CLT needs finite variance.** Averages of heavy-tailed (e.g. Cauchy) variables do
  *not* converge to a Gaussian.

In [ ]:
# density can exceed 1
print('N(0, 0.1) peak density =', round(stats.norm.pdf(0, 0, 0.1), 2), '(> 1)')

# population vs sample variance on a small sample
x = rng.normal(0, 1, 8)
print('var ddof=0 (population) =', round(x.var(), 3),
      '| ddof=1 (sample) =', round(x.var(ddof=1), 3))

# log-likelihood at the boundary
print('log-likelihood term log(theta) at theta -> 0 :', np.log(1e-300), '(heads toward -inf)')

**What to notice:** the narrow Gaussian's peak density is ~4, not a probability; the two
variance conventions differ noticeably at `n = 8` (dividing by 7 vs 8); and `log θ` dives
toward `−∞` at the boundary — the reasons real code uses `ddof=1` for sample variance and
guards log-likelihoods away from 0 and 1.

## Key takeaways

- **PMF (discrete) is probability; PDF (continuous) is density** — density can exceed 1,
  only its area is a probability.
- Bernoulli/Binomial (`E=p`/`np`), Gaussian (`E=μ, Var=σ²`), Uniform, Poisson — know the
  moments; they match heavy sampling exactly.
- The **Gaussian normalizer** `1/(σ√2π)` is what makes the bell curve integrate to 1; the
  **CLT** makes the Gaussian the default for aggregated noise.
- **Maximum likelihood** picks the parameter maximizing the data's probability; for a
  coin it's `s/n`, and `scipy.stats.<dist>.fit` does it for you.
- Watch `ddof` for variance, `log(0)` in likelihoods, and the CLT's finite-variance need.

**Next:** [Maximum Likelihood Estimation](https://ml-viz-ruby.vercel.app/courses/probability-statistics/03-maximum-likelihood-estimation).

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — The Gaussian PDF

Write the bell curve yourself:

$$\mathcal{N}(x \mid \mu, \sigma^2) = \frac{1}{\sigma\sqrt{2\pi}} \, \exp\!\left(-\frac{1}{2}\left(\frac{x - \mu}{\sigma}\right)^{\!2}\right)$$

The checks verify the peak height $1/(\sigma\sqrt{2\pi})$, symmetry around $\mu$, that the density integrates to 1, and the area-stays-1 trade-off: **halving $\sigma$ doubles the peak**.

In [ ]:
def gaussian_pdf(x, mu, sigma):
    """Normal density at x (x may be a scalar or an array)."""
    x = np.asarray(x, dtype=float)

    # TODO(you): the standardized distance z = (x - mu) / sigma
    z = ...

    # TODO(you): exp(-z^2 / 2) divided by (sigma * sqrt(2 pi))
    return ...

In [ ]:
assert abs(gaussian_pdf(0, 0, 1) - 1 / np.sqrt(2 * np.pi)) < 1e-12, "standard normal peaks at ~0.3989"
assert abs(gaussian_pdf(3, 1, 2) - gaussian_pdf(-1, 1, 2)) < 1e-12, "symmetric around mu"

xs = np.linspace(-10, 10, 20001)
assert abs(np.trapezoid(gaussian_pdf(xs, 0.5, 1.3), xs) - 1) < 1e-6, "a PDF must integrate to 1"

assert abs(gaussian_pdf(2, 2, 0.5) - 2 * gaussian_pdf(0, 0, 1)) < 1e-12, "halving sigma doubles the peak"

# Edge cases: extreme values and a near-degenerate (tiny-variance) Gaussian
assert gaussian_pdf(50, 0, 1) < 1e-10, "far in the tail the density is essentially 0"
sigma_tiny = 1e-3
assert gaussian_pdf(0, 0, sigma_tiny) > 300, "a tiny sigma concentrates mass into a tall spike at mu"
assert gaussian_pdf(1, 0, sigma_tiny) < 1e-6, "...and almost none a whole unit away"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def gaussian_pdf(x, mu, sigma):
    x = np.asarray(x, dtype=float)
    z = (x - mu) / sigma
    return np.exp(-0.5 * z ** 2) / (sigma * np.sqrt(2 * np.pi))
```

</details>

### Exercise 2 — The Binomial PMF

The probability of exactly $k$ successes in $n$ independent trials combines *how many ways* with *how likely each way is*:

$$P(X = k) = \binom{n}{k} \, p^k \, (1-p)^{n-k}$$

Use `math.comb(n, k)` for the binomial coefficient. The checks confirm the PMF sums to 1 and that the mean comes out to $np$.

In [ ]:
import math


def binomial_pmf(k, n, p):
    """P(exactly k successes in n trials with success probability p)."""
    # TODO(you): the three factors: math.comb(n, k), p**k, (1-p)**(n-k)
    return ...

In [ ]:
assert abs(binomial_pmf(1, 2, 0.5) - 0.5) < 1e-12, "two fair flips: P(exactly 1 head) = 1/2"
assert abs(sum(binomial_pmf(k, 10, 0.3) for k in range(11)) - 1) < 1e-12, "the PMF must sum to 1"

mean = sum(k * binomial_pmf(k, 10, 0.3) for k in range(11))
assert abs(mean - 3.0) < 1e-12, "E[X] = n*p = 3"
assert abs(binomial_pmf(0, 5, 0.2) - 0.8 ** 5) < 1e-12, "zero successes = (1-p)^n"

# Edge cases: degenerate (p=0, p=1) and the n=1 Bernoulli special case
assert abs(binomial_pmf(0, 5, 0.0) - 1.0) < 1e-12, "p=0: a degenerate distribution, all mass at k=0"
assert abs(binomial_pmf(5, 5, 1.0) - 1.0) < 1e-12, "p=1: a degenerate distribution, all mass at k=n"
assert abs(binomial_pmf(1, 1, 0.3) - 0.3) < 1e-12, "n=1 is just a Bernoulli trial"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def binomial_pmf(k, n, p):
    return math.comb(n, k) * p ** k * (1 - p) ** (n - k)
```

</details>

---
## Extra practice — the discrete/continuous distribution bank

Three more distributions worth having in your fingers, taken from the
[DML-OpenProblem](https://github.com/Open-Deep-ML/DML-OpenProblem) problem bank:
**binomial** (`79_binomial-distribution-probability`), **normal PDF**
(`80_normal-distribution-pdf-calculator` — the same Gaussian from Exercise 1,
but with DML's exact `(x, mean, std_dev)` signature and 5-decimal rounding),
and **Poisson** (`81_poisson-distribution-probability-calculator`, brand new
here — the distribution of counts of rare events in a fixed interval).

$$P(X=k) = \binom{n}{k}p^k(1-p)^{n-k} \qquad
\mathcal N(x\mid\mu,\sigma) = \frac{1}{\sigma\sqrt{2\pi}}e^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2}
\qquad
P(X=k) = \frac{\lambda^k e^{-\lambda}}{k!}$$

In [ ]:
import math


def binomial_probability(n, k, p):
    """DML 79 — P(exactly k successes in n trials), rounded to 5 decimals."""
    # TODO(you): math.comb(n, k) * p**k * (1 - p)**(n - k), rounded to 5 places
    return ...


def normal_pdf(x, mean, std_dev):
    """DML 80 — the Gaussian density, rounded to 5 decimals (DML's exact signature)."""
    # TODO(you): same formula as gaussian_pdf above, just rounded to 5 places
    return ...


def poisson_probability(k, lam):
    """DML 81 — P(exactly k events) for a Poisson(lam), rounded to 5 decimals."""
    # TODO(you): lam**k * exp(-lam) / k!, rounded to 5 places
    return ...

In [ ]:
# Checks — run me

# DML's own worked examples
assert abs(binomial_probability(6, 2, 0.5) - 0.23438) < 1e-9, "DML 79 example: n=6, k=2, p=0.5"
assert abs(normal_pdf(16, 15, 2.04) - 0.17342) < 1e-9, "DML 80 example: x=16, mean=15, std_dev=2.04"
assert abs(poisson_probability(3, 5) - 0.14037) < 1e-9, "DML 81 example: k=3, lam=5"

# Edge cases: degenerate distributions
assert abs(binomial_probability(5, 0, 0.0) - 1.0) < 1e-9, "p=0 -> certainty of 0 successes"
assert abs(binomial_probability(5, 5, 1.0) - 1.0) < 1e-9, "p=1 -> certainty of 5 successes"
assert abs(poisson_probability(0, 0) - 1.0) < 1e-9, "lam=0 -> certainty of 0 events"
assert abs(poisson_probability(1, 0) - 0.0) < 1e-9, "lam=0 -> impossible to see any events"

# Sanity check against the plotting code from section 2 above (scipy is already imported there) —
# these are implemented from scratch; scipy is used only to double-check the numbers, never to compute them.
print('binomial: ours={:.5f}  scipy={:.5f}'.format(binomial_probability(6, 2, 0.5), stats.binom.pmf(2, 6, 0.5)))
print('normal:   ours={:.5f}  scipy={:.5f}'.format(normal_pdf(16, 15, 2.04), stats.norm.pdf(16, 15, 2.04)))
print('poisson:  ours={:.5f}  scipy={:.5f}'.format(poisson_probability(3, 5), stats.poisson.pmf(3, 5)))
print("✅ Extra practice passed")

<details>
<summary>💡 Show solution</summary>

```python
def binomial_probability(n, k, p):
    return round(math.comb(n, k) * p ** k * (1 - p) ** (n - k), 5)


def normal_pdf(x, mean, std_dev):
    z = (x - mean) / std_dev
    return round(math.exp(-0.5 * z ** 2) / (std_dev * math.sqrt(2 * math.pi)), 5)


def poisson_probability(k, lam):
    return round((lam ** k) * math.exp(-lam) / math.factorial(k), 5)
```

</details>